# Três testes manuais: API OCDS + dados vindos no CSV do PNCP
Cada caso declara `buyerID`, `releaseStartDate` e `releaseEndDate`. A função salva o JSON bruto da página em `analise-inicial-csv-e-api/docs/output/` e mantém o retorno em `resposta_1`, `resposta_2` ou `resposta_3`.

`offSet` e `page` seguem o valor padrão da URL no Swagger.

In [1]:
import json
from consulta_api import consultar_ocds, url_ocds
from pathlib import Path

BASE_DIR = globals().get("BASE_DIR", Path.cwd().resolve())

def executar_teste(buyerID, releaseStartDate, releaseEndDate, nome_arquivo, page=1, offSet=10):
    filtros = dict(buyer_id=buyerID, release_start_date=releaseStartDate,
                   release_end_date=releaseEndDate, page=page, offset=offSet) # {"page": 1, "offset": 10, etc...}
    print('GET', url_ocds(**filtros)) # desempacota os dados do dicionário filtros e passa como argumentos para a função url_ocds
    resposta = consultar_ocds(**filtros) # desempacota os dados do dicionário filtros e passa como argumentos para a função consultar_ocds
    releases = resposta.get('releases')
    if isinstance(releases, list):
        print('Releases nesta página:', len(releases))
    output_dir = BASE_DIR / 'docs' / 'output' 
    output_dir.mkdir(exist_ok=True)
    caminho = output_dir / nome_arquivo
    conteudo = json.dumps(resposta, ensure_ascii=False, indent=2, allow_nan=False) + '\n'
    caminho.write_text(conteudo, encoding='utf-8')
    print('Salvo:', caminho)
    return resposta


## 1. SES/DF — contratações
Reproduz os filtros do link já testado no navegador. Publicação: 01/09/2026 a 30/09/2026.

In [2]:
buyerID = '00394700000108'
releaseStartDate = '2026-09-01'
releaseEndDate = '2026-09-30'

resposta_1 = executar_teste(buyerID, releaseStartDate, releaseEndDate, 'resposta_1.json')

GET https://dadosabertos.compras.gov.br/modulo-ocds/1_releases?page=1&offSet=10&buyerID=00394700000108&releaseStartDate=2026-09-01&releaseEndDate=2026-09-30
Releases nesta página: 10
Salvo: C:\Users\gabri\OneDrive\Área de Trabalho\joao\TB\medicamentos-fdcc\tasks\analise-inicial-csv-e-api\docs\output\resposta_1.json


## 2. PMDF — insulina FIASP, dispensa, com resultado indicado no CSV
Publicação: 28/07/2026. O intervalo estreito reduz o volume da resposta.


In [3]:
buyerID = '08942610000116'
releaseStartDate = '2026-07-27'
releaseEndDate = '2026-07-29'

resposta_2 = executar_teste(buyerID, releaseStartDate, releaseEndDate, 'resposta_2.json')

GET https://dadosabertos.compras.gov.br/modulo-ocds/1_releases?page=1&offSet=10&buyerID=08942610000116&releaseStartDate=2026-07-27&releaseEndDate=2026-07-29
Releases nesta página: 3
Salvo: C:\Users\gabri\OneDrive\Área de Trabalho\joao\TB\medicamentos-fdcc\tasks\analise-inicial-csv-e-api\docs\output\resposta_2.json


## 3. SES/DF — estradiol, pregão para demanda judicial, com resultado indicado no CSV
Publicação: 14/04/2026. “Demanda judicial” é contexto descrito no objeto, não modalidade.


In [4]:
buyerID = '00394700000108'
releaseStartDate = '2026-04-13'
releaseEndDate = '2026-04-15'

resposta_3 = executar_teste(buyerID, releaseStartDate, releaseEndDate, 'resposta_3.json')

GET https://dadosabertos.compras.gov.br/modulo-ocds/1_releases?page=1&offSet=10&buyerID=00394700000108&releaseStartDate=2026-04-13&releaseEndDate=2026-04-15
Releases nesta página: 10
Salvo: C:\Users\gabri\OneDrive\Área de Trabalho\joao\TB\medicamentos-fdcc\tasks\analise-inicial-csv-e-api\docs\output\resposta_3.json


## O que conferir nos JSONs
1. `releases[].buyer`/`parties`: comprador; `tender`: objeto, modalidade, itens e estimativa, **se presentes**.
2. `awards`: itens, valores, fornecedores e datas, **se presentes**. Conferir a qual item cada fornecedor/valor pertence.
3. Comparar identificadores da resposta com o número PNCP do CSV; não assumir que a primeira release é a linha escolhida.
4. Conferir a definição da data encontrada. `releaseStartDate` e `releaseEndDate` filtram publicação, não homologação.
5. Se a contratação não aparecer na primeira página, consultar as páginas seguintes ou ampliar o intervalo; resposta vazia não comprova ausência da compra.
